# Research notebook archive

Original project notebook with outputs and personal metadata removed. Existing Colab paths and experiment settings require configuration; see the repository README before running.

# Stage 1 — Tier 1 metrics: per-class fidelity tables

Reads the scored parquets written by `stage_1.ipynb` (with PriDe debiasing)
and produces the **headline per-class fidelity table** specified in §2.2 of
the report. No model inference here — this is pure analysis.

For each target $Y$ and each class $c$, we report:

- **Argmax accuracy** $\widehat{\text{acc}}_c$ (point estimate + bootstrap CI)
- **Per-class precision, recall, F1** (point estimate + bootstrap CI)
- **Cohen's $\kappa$** (chance-corrected agreement; one scalar per target)
- **Macro and weighted F1** (summary rows in the per-target table)
- **Brier score** $B_c(Y)$ (eq. 5; point estimate + bootstrap CI)
- **Clipped log-likelihood** $L^{\text{clip}}_c(Y)$ with $\log \epsilon = -10$
- **Clipping rate**: fraction of rows in class $c$ where the floor was active
- **Random baseline** $1 - 1/n_Y$ for Brier; $1/n_Y$ for accuracy
- **Population row** (Hájek-weighted): population Brier, $N_{\text{eff}}$, weighted accuracy

Confidence intervals use **stratified bootstrap with $B = 10{,}000$
replicates**: profiles are resampled with replacement *within each class*
(preserving the stratified design), the metric is recomputed on each
replicate, and we report empirical 2.5th and 97.5th percentiles.

Headline metrics are computed on **debiased** probabilities (canonical
per the report's Results section). Raw probabilities are computed
side-by-side and reported as an ablation in the appendix.

**Section overview**

| §  | Output                                                          |
|----|-----------------------------------------------------------------|
| 1  | Setup: load all 7 scored parquets                               |
| 2  | Metric implementations (Brier, clipped LL, accuracy)            |
| 3  | Stratified bootstrap                                            |
| 4  | Per-target fidelity tables (raw + debiased side-by-side)        |
| 5  | Cross-target summary table                                      |
| 6  | Headline figure: per-class Brier with CIs                       |
| 7  | Raw-vs-debiased ablation summary                                |


## 1. Setup

Load the per-target scored parquet + metadata sidecar. Same loader as the
debiasing-eval notebook so legacy parquets degrade gracefully.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import json
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# --- Model under analysis -------------------------------------------------
# MODEL_NAME = "mistralai/Ministral-3-14B-Instruct-2512"
#MODEL_NAME = "Qwen/Qwen3-30B-A3B-Instruct-2507"
#MODEL_NAME = "Qwen3-4B-Instruct-2507-FT_cumulative_bf16_05_07"
#MODEL_NAME = "Qwen/Qwen3-4B-Instruct-2507_05_05"

#MODEL_NAME = "Qwen2.5-14B-Instruct-FT_natural_canonical_step07_bf16_test_balanced_05_11"
# MODEL_NAME = "Qwen2.5-14B-Instruct_baseline_test_balanced_05_11"

# MODEL_NAME = "Qwen2.5-14B-Instruct-FT_natural_canonical_step07_bf16_test_raw_05_12_lauren"
#MODEL_NAME = "Qwen2.5-14B-Instruct-FT_balanced_canonical_step07_bf16_test_raw_05_12"
MODEL_NAME = "Qwen2.5-14B-Instruct_baseline_test_raw_05_12"

# --- Display knobs --------------------------------------------------------
pd.set_option("display.precision", 3)
pd.set_option("display.max_columns", 50)
plt.rcParams["figure.dpi"] = 110
sns.set_style("whitegrid")

# --- Bootstrap & metric config -------------------------------------------
GLOBAL_SEED = 42
B_BOOT      = 10_000          # §2.2 pre-specification
LOG_EPS     = -10.0           # clipped log-likelihood floor
CI_ALPHA    = 0.05            # 95% percentile CIs

# --- I/O ------------------------------------------------------------------
SCORED_DIR  = Path(
    f"/content/drive/MyDrive/NLP_project/data/"
    f"stage_1_output_{MODEL_NAME}/"
)
METRICS_DIR = Path(
    f"/content/drive/MyDrive/NLP_project/data/"
    f"stage_1_output_{MODEL_NAME}/phase1_metrics/"
)
METRICS_DIR.mkdir(parents=True, exist_ok=True)

# --- Targets to analyze ---------------------------------------------------
TARGETS = [
    "SEX",
    "AGE_BINS",
    "RACE_ETHNICITY",
    "EDUCATION_LEVEL",
    "INCTOT_BINS",
    "HEALTH_INSURANCE",
    "ENGLISH_STATUS",
]


In [ ]:
def detect_prob_kinds(scored: pd.DataFrame, classes: list) -> list:
    """Return ['raw', 'deb'] when both are present, else ['raw']."""
    has_raw = all(f"prob_raw_{c}" in scored.columns for c in classes)
    has_deb = all(f"prob_deb_{c}" in scored.columns for c in classes)
    if has_raw and has_deb:
        return ["raw", "deb"]
    return ["raw"]


def load_target(target: str) -> dict:
    """Load scored parquet + metadata for one target."""
    pq_path  = SCORED_DIR / f"phase1_scored_{target}.parquet"
    meta_path = SCORED_DIR / f"phase1_scored_{target}.meta.json"

    if not pq_path.exists():
        raise FileNotFoundError(f"Missing scored parquet for {target}: {pq_path}")
    if not meta_path.exists():
        raise FileNotFoundError(f"Missing metadata for {target}: {meta_path}")

    scored = pd.read_parquet(pq_path)
    with open(meta_path) as f:
        meta = json.load(f)

    classes = meta["class_labels"]
    prob_kinds = detect_prob_kinds(scored, classes)

    # Legacy fallback: legacy parquets only have prob_{c} (which is raw
    # because legacy = pre-PriDe). Alias to prob_raw_{c} for uniform access.
    needs_legacy_alias = (
        prob_kinds == ["raw"]
        and not all(f"prob_raw_{c}" in scored.columns for c in classes)
    )
    if needs_legacy_alias:
        for c in classes:
            scored[f"prob_raw_{c}"] = scored[f"prob_{c}"]

    return {
        "scored":        scored,
        "meta":          meta,
        "classes":       classes,
        "prob_kinds":    prob_kinds,
        "applied_pride": meta.get("apply_pride", False),
    }


def prob_cols(classes: list, kind: str = "deb") -> list:
    return [f"prob_{kind}_{c}" for c in classes]


runs = {}
for t in TARGETS:
    try:
        runs[t] = load_target(t)
        n = len(runs[t]["scored"])
        k = len(runs[t]["classes"])
        kinds = "+".join(runs[t]["prob_kinds"])
        pride = "PriDe" if runs[t]["applied_pride"] else "raw-only"
        print(f"  {t:<20} loaded: {n:,} rows, {k} classes, "
              f"kinds=[{kinds}] ({pride})")
    except FileNotFoundError as e:
        warnings.warn(f"Skipping {t}: {e}")

print(f"\nLoaded {len(runs)} / {len(TARGETS)} targets.")

# Default kind for headline metrics
ALL_HAVE_DEB = all("deb" in r["prob_kinds"] for r in runs.values())
DEFAULT_KIND = "deb" if ALL_HAVE_DEB else "raw"
print(f"Default kind for headline: {DEFAULT_KIND!r}")


## 2. Metric implementations

Vectorized per-row metric computations. All take a probability matrix
$P \in \mathbb{R}^{N \times n_Y}$ (one row per profile, one column per
class in `classes` order) and a true-class vector $y$ as integer indices,
and return per-row metric values that downstream code aggregates by
class or by Hájek weight.

The per-row primitives are:

- **Brier**: $b_i = \sum_k (P_{ik} - \mathbb{1}[y_i = k])^2 \in [0, 2]$
- **Clipped LL**: $\ell_i = \max(\log P_{i, y_i}, \log \epsilon)$
- **Argmax correctness**: $\mathbb{1}[\arg\max_k P_{ik} = y_i]$


In [ ]:
def brier_per_row(P: np.ndarray, y: np.ndarray) -> np.ndarray:
    """Per-row Brier score. P: (N, n_Y), y: (N,) integer class indices."""
    N, n_Y = P.shape
    onehot = np.zeros_like(P)
    onehot[np.arange(N), y] = 1.0
    return np.sum((P - onehot) ** 2, axis=1)


def clipped_ll_per_row(P: np.ndarray, y: np.ndarray,
                       log_eps: float = LOG_EPS) -> tuple:
    """Per-row clipped log-likelihood, plus a per-row clipping flag.

    Returns (ll, clipped) where:
      ll[i]      = max(log P[i, y[i]], log_eps)
      clipped[i] = True if the floor was active for row i
    """
    p_true = P[np.arange(len(y)), y]
    # Avoid -inf from log(0) by clipping probabilities to a tiny positive
    # value before taking log; the eps floor handles meaningfully small
    # but nonzero probabilities.
    with np.errstate(divide="ignore"):
        raw_ll = np.where(p_true > 0, np.log(p_true), -np.inf)
    clipped = raw_ll < log_eps
    ll = np.where(clipped, log_eps, raw_ll)
    return ll, clipped


def argmax_correct_per_row(P: np.ndarray, y: np.ndarray) -> np.ndarray:
    """Per-row argmax correctness as 0/1 ints."""
    return (np.argmax(P, axis=1) == y).astype(np.int8)


def assemble_prob_matrix(scored: pd.DataFrame, classes: list,
                         kind: str) -> np.ndarray:
    """Stack prob_{kind}_{c} columns into an (N, n_Y) matrix."""
    return scored[prob_cols(classes, kind)].to_numpy()


def true_indices(scored: pd.DataFrame, target_col: str,
                 classes: list) -> np.ndarray:
    """Map true_{target} string labels to integer class indices."""
    cls_to_idx = {c: i for i, c in enumerate(classes)}
    return scored[f"true_{target_col}"].map(cls_to_idx).to_numpy()


## 3. Stratified bootstrap

For per-class metrics we resample within-stratum (within each true-class
bucket). For population-level metrics we still resample within stratum
(preserving the design), then aggregate using the original Hájek weights —
this gives correct CIs for both estimands under the same resampling scheme.

The bootstrap operates on **per-row metric values** plus indices, not on
raw probabilities, which is much faster: we precompute $b_i$ and $\ell_i$
once and then index into them $B$ times.

We also report the **per-class effective sample size** for each estimate
(here just $n_c$ since per-class metrics are unweighted within stratum)
and the population-level $N_{\text{eff}} = (\sum w_i)^2 / \sum w_i^2$.


In [ ]:
def stratified_bootstrap_indices(true_idx: np.ndarray, n_classes: int,
                                  B: int, seed: int) -> np.ndarray:
    """Generate B sets of bootstrap indices, resampling within each class.

    Returns an array of shape (B, N) where row b is a bootstrap permutation
    of the original row indices that preserves per-class counts.
    """
    N = len(true_idx)
    rng = np.random.default_rng(seed)

    # Index pools per class
    class_pools = [np.where(true_idx == c)[0] for c in range(n_classes)]

    boot_idx = np.empty((B, N), dtype=np.int64)
    cursor = np.zeros(N, dtype=np.int64)  # not used, just for shape clarity
    # Fill buffer class-by-class so each bootstrap row preserves stratum sizes
    out_pos = 0
    for c, pool in enumerate(class_pools):
        n_c = len(pool)
        if n_c == 0:
            continue
        # Vectorized: draw (B, n_c) indices into pool for this class
        sampled = rng.integers(0, n_c, size=(B, n_c))
        boot_idx[:, out_pos:out_pos + n_c] = pool[sampled]
        out_pos += n_c
    return boot_idx


def bootstrap_per_class_metric(per_row: np.ndarray, true_idx: np.ndarray,
                                boot_idx: np.ndarray,
                                n_classes: int) -> np.ndarray:
    """Compute per-class means under each of B bootstrap resamples.

    Returns array of shape (B, n_classes).
    """
    B = boot_idx.shape[0]
    out = np.zeros((B, n_classes), dtype=np.float64)
    # For each class, the rows in boot_idx that belong to that class are
    # those with true_idx[boot_idx] == c. Compute via fancy indexing.
    boot_y = true_idx[boot_idx]                # (B, N)
    boot_v = per_row[boot_idx]                 # (B, N)
    for c in range(n_classes):
        mask = (boot_y == c)
        # Avoid division by zero for empty classes (shouldn't happen given
        # within-stratum resampling, but defensive).
        n_c = mask.sum(axis=1)
        sums = (boot_v * mask).sum(axis=1)
        out[:, c] = np.where(n_c > 0, sums / np.maximum(n_c, 1), np.nan)
    return out


def bootstrap_population_metric(per_row: np.ndarray, weights: np.ndarray,
                                 boot_idx: np.ndarray) -> np.ndarray:
    """Hajek-weighted population mean under each of B bootstrap resamples.

    Returns array of shape (B,).
    """
    boot_v = per_row[boot_idx]
    boot_w = weights[boot_idx]
    return (boot_v * boot_w).sum(axis=1) / boot_w.sum(axis=1)


def percentile_ci(samples: np.ndarray, alpha: float = CI_ALPHA) -> tuple:
    """Empirical percentile CI from bootstrap samples."""
    lo = np.nanpercentile(samples, 100 * alpha / 2,        axis=0)
    hi = np.nanpercentile(samples, 100 * (1 - alpha / 2),  axis=0)
    return lo, hi


def n_eff(weights: np.ndarray) -> float:
    """Hajek effective sample size: (sum w)^2 / sum w^2."""
    s1 = float(weights.sum())
    s2 = float((weights ** 2).sum())
    return s1 * s1 / s2 if s2 > 0 else 0.0


In [ ]:
# ---------------------------------------------------------------------------
# Argmax-prediction primitives (precision, recall, F1, Cohen's kappa)
# ---------------------------------------------------------------------------
#
# These take an integer prediction vector `pred` and true-class vector `y` (both
# of shape (N,) with values in [0, n_Y)) and return per-class or scalar metrics.
# All bootstrap variants reuse `stratified_bootstrap_indices` from the cell
# above so CIs are consistent with the Brier / accuracy CIs already produced.
#
# Empty-class convention: when a class has zero predicted positives in a
# bootstrap replicate, precision is undefined; we set precision = 0 and F1 = 0
# in that replicate (sklearn `zero_division=0` convention).

def confusion_counts(pred: np.ndarray, y: np.ndarray, n_classes: int) -> dict:
    """Return TP, FP, FN per class as length-n_classes arrays."""
    TP = np.zeros(n_classes, dtype=np.int64)
    FP = np.zeros(n_classes, dtype=np.int64)
    FN = np.zeros(n_classes, dtype=np.int64)
    for c in range(n_classes):
        pred_pos  = (pred == c)
        truth_pos = (y == c)
        TP[c] = int(np.sum(pred_pos & truth_pos))
        FP[c] = int(np.sum(pred_pos & ~truth_pos))
        FN[c] = int(np.sum(~pred_pos & truth_pos))
    return {"TP": TP, "FP": FP, "FN": FN}


def precision_recall_f1(pred: np.ndarray, y: np.ndarray,
                         n_classes: int) -> dict:
    """Per-class precision, recall, F1 + macro and weighted averages.

    Returns dict with keys:
      precision, recall, f1     : (n_classes,) arrays
      macro_precision, macro_recall, macro_f1     : scalars
      weighted_precision, weighted_recall, weighted_f1 : scalars (weighted by
                                                         true-class frequency)
    """
    cc = confusion_counts(pred, y, n_classes)
    TP, FP, FN = cc["TP"], cc["FP"], cc["FN"]

    # Per-class precision, recall, F1 with zero-division guard.
    with np.errstate(divide="ignore", invalid="ignore"):
        precision = np.where((TP + FP) > 0, TP / (TP + FP), 0.0)
        recall    = np.where((TP + FN) > 0, TP / (TP + FN), 0.0)
        denom = precision + recall
        f1 = np.where(denom > 0, 2 * precision * recall / denom, 0.0)

    # Macro: unweighted mean across classes.
    macro_p = float(np.mean(precision))
    macro_r = float(np.mean(recall))
    macro_f = float(np.mean(f1))

    # Weighted: weight by true-class frequency (i.e. support).
    support = TP + FN
    total = support.sum()
    if total > 0:
        w = support / total
        weighted_p = float(np.sum(w * precision))
        weighted_r = float(np.sum(w * recall))
        weighted_f = float(np.sum(w * f1))
    else:
        weighted_p = weighted_r = weighted_f = float("nan")

    return {
        "precision": precision, "recall": recall, "f1": f1,
        "macro_precision": macro_p, "macro_recall": macro_r, "macro_f1": macro_f,
        "weighted_precision": weighted_p, "weighted_recall": weighted_r,
        "weighted_f1": weighted_f,
    }


def cohens_kappa(pred: np.ndarray, y: np.ndarray, n_classes: int) -> float:
    """Unweighted Cohen's kappa.

    kappa = (p_o - p_e) / (1 - p_e)  where p_o is observed agreement and
    p_e is chance agreement under the marginal product distribution.
    Returns 0.0 if 1 - p_e <= 0 (degenerate: one rater puts everything in
    one class and the other agrees).
    """
    N = len(y)
    if N == 0:
        return float("nan")

    p_o = float(np.mean(pred == y))

    # Marginal frequencies for predictor and truth.
    pred_marg  = np.bincount(pred, minlength=n_classes) / N
    truth_marg = np.bincount(y,    minlength=n_classes) / N

    p_e = float(np.sum(pred_marg * truth_marg))
    denom = 1.0 - p_e
    if denom <= 0:
        return 0.0
    return (p_o - p_e) / denom


def bootstrap_classification_metrics(pred: np.ndarray, y: np.ndarray,
                                       n_classes: int, boot_idx: np.ndarray
                                       ) -> dict:
    """Run B bootstrap replicates of precision / recall / F1 / kappa.

    Returns dict with:
      precision  : (B, n_classes)
      recall     : (B, n_classes)
      f1         : (B, n_classes)
      macro_precision, macro_recall, macro_f1     : (B,)
      weighted_precision, weighted_recall, weighted_f1 : (B,)
      kappa      : (B,)
    """
    B = boot_idx.shape[0]
    precision = np.zeros((B, n_classes))
    recall    = np.zeros((B, n_classes))
    f1        = np.zeros((B, n_classes))
    macro_p   = np.zeros(B)
    macro_r   = np.zeros(B)
    macro_f   = np.zeros(B)
    weighted_p = np.zeros(B)
    weighted_r = np.zeros(B)
    weighted_f = np.zeros(B)
    kappa     = np.zeros(B)

    for b in range(B):
        idx = boot_idx[b]
        pred_b = pred[idx]
        y_b    = y[idx]
        prf = precision_recall_f1(pred_b, y_b, n_classes)
        precision[b]  = prf["precision"]
        recall[b]     = prf["recall"]
        f1[b]         = prf["f1"]
        macro_p[b]    = prf["macro_precision"]
        macro_r[b]    = prf["macro_recall"]
        macro_f[b]    = prf["macro_f1"]
        weighted_p[b] = prf["weighted_precision"]
        weighted_r[b] = prf["weighted_recall"]
        weighted_f[b] = prf["weighted_f1"]
        kappa[b]      = cohens_kappa(pred_b, y_b, n_classes)

    return {
        "precision": precision, "recall": recall, "f1": f1,
        "macro_precision": macro_p, "macro_recall": macro_r, "macro_f1": macro_f,
        "weighted_precision": weighted_p, "weighted_recall": weighted_r,
        "weighted_f1": weighted_f,
        "kappa": kappa,
    }


## 4. Per-target fidelity tables

For each (target, kind), we compute:

- One row per class with: $n_c$, accuracy [CI], Brier [CI], clipped LL,
  clipping rate
- One **population** row with: $N_{\text{eff}}$, weighted accuracy [CI],
  population Brier [CI], population clipped LL
- One **uniform baseline** row with: $1/n_Y$ accuracy, $1 - 1/n_Y$ Brier,
  $-\log n_Y$ LL

Output: one DataFrame per (target, kind), persisted as CSV.


In [ ]:
def compute_fidelity_table(scored: pd.DataFrame, target_col: str,
                            classes: list, kind: str,
                            B: int = B_BOOT,
                            seed: int = GLOBAL_SEED) -> pd.DataFrame:
    """Build the per-class + MACRO + WEIGHTED + POPULATION + BASELINE table.

    Per-class columns include:
      n_c, accuracy [CI], brier [CI], clipped_ll [CI], clip_rate,
      precision [CI], recall [CI], f1 [CI].

    Summary rows below the per-class block:
      MACRO_AVG         : unweighted mean of precision / recall / F1
                          (the "macro" aggregation; the cells in this row
                          for accuracy/brier/clipped_ll are NaN by design).
      WEIGHTED_AVG      : true-class-frequency-weighted mean of P/R/F1.
      POPULATION        : Hajek-weighted population mean of accuracy / brier /
                          clipped_ll plus the population-level kappa.
      UNIFORM_BASELINE  : 1/n_Y reference.
    """
    n_Y = len(classes)
    P = assemble_prob_matrix(scored, classes, kind)
    y = true_indices(scored, target_col, classes)
    w = scored["hajek_weight"].to_numpy() if "hajek_weight" in scored else         np.ones(len(scored))

    # Per-row primitives (computed once).
    b = brier_per_row(P, y)
    ll, clipped = clipped_ll_per_row(P, y, LOG_EPS)
    correct = argmax_correct_per_row(P, y)
    pred = np.argmax(P, axis=1)   # used for precision / recall / F1 / kappa

    # Bootstrap indices (one set, reused across all metrics).
    boot_idx = stratified_bootstrap_indices(y, n_Y, B, seed)

    # ---- Per-class point estimates (probability-based metrics) ----------
    per_class_n  = np.array([(y == c).sum() for c in range(n_Y)])
    per_class_acc = np.array([
        correct[y == c].mean() if (y == c).any() else np.nan
        for c in range(n_Y)
    ])
    per_class_brier = np.array([
        b[y == c].mean() if (y == c).any() else np.nan
        for c in range(n_Y)
    ])
    per_class_ll = np.array([
        ll[y == c].mean() if (y == c).any() else np.nan
        for c in range(n_Y)
    ])
    per_class_clip_rate = np.array([
        clipped[y == c].mean() if (y == c).any() else np.nan
        for c in range(n_Y)
    ])

    # ---- Bootstrap CIs for probability-based metrics --------------------
    boot_acc   = bootstrap_per_class_metric(correct.astype(float), y, boot_idx, n_Y)
    boot_brier = bootstrap_per_class_metric(b, y, boot_idx, n_Y)
    boot_ll    = bootstrap_per_class_metric(ll, y, boot_idx, n_Y)
    acc_lo,   acc_hi   = percentile_ci(boot_acc)
    brier_lo, brier_hi = percentile_ci(boot_brier)
    ll_lo,    ll_hi    = percentile_ci(boot_ll)

    # ---- Classification metrics (precision / recall / F1 / kappa) -------
    cls_pt = precision_recall_f1(pred, y, n_Y)
    kappa_pt = cohens_kappa(pred, y, n_Y)
    cls_boot = bootstrap_classification_metrics(pred, y, n_Y, boot_idx)

    prec_lo,  prec_hi = percentile_ci(cls_boot["precision"])
    rec_lo,   rec_hi  = percentile_ci(cls_boot["recall"])
    f1_lo,    f1_hi   = percentile_ci(cls_boot["f1"])

    macro_p_lo, macro_p_hi = percentile_ci(cls_boot["macro_precision"][:, None])
    macro_p_lo, macro_p_hi = float(macro_p_lo[0]), float(macro_p_hi[0])
    macro_r_lo, macro_r_hi = percentile_ci(cls_boot["macro_recall"][:, None])
    macro_r_lo, macro_r_hi = float(macro_r_lo[0]), float(macro_r_hi[0])
    macro_f_lo, macro_f_hi = percentile_ci(cls_boot["macro_f1"][:, None])
    macro_f_lo, macro_f_hi = float(macro_f_lo[0]), float(macro_f_hi[0])

    weighted_p_lo, weighted_p_hi = percentile_ci(cls_boot["weighted_precision"][:, None])
    weighted_p_lo, weighted_p_hi = float(weighted_p_lo[0]), float(weighted_p_hi[0])
    weighted_r_lo, weighted_r_hi = percentile_ci(cls_boot["weighted_recall"][:, None])
    weighted_r_lo, weighted_r_hi = float(weighted_r_lo[0]), float(weighted_r_hi[0])
    weighted_f_lo, weighted_f_hi = percentile_ci(cls_boot["weighted_f1"][:, None])
    weighted_f_lo, weighted_f_hi = float(weighted_f_lo[0]), float(weighted_f_hi[0])

    kappa_lo, kappa_hi = percentile_ci(cls_boot["kappa"][:, None])
    kappa_lo, kappa_hi = float(kappa_lo[0]), float(kappa_hi[0])

    rows = []
    for c_idx, c in enumerate(classes):
        rows.append({
            "class":         c,
            "n_c":           int(per_class_n[c_idx]),
            "accuracy":      per_class_acc[c_idx],
            "acc_lo":        acc_lo[c_idx],
            "acc_hi":        acc_hi[c_idx],
            "brier":         per_class_brier[c_idx],
            "brier_lo":      brier_lo[c_idx],
            "brier_hi":      brier_hi[c_idx],
            "clipped_ll":    per_class_ll[c_idx],
            "clip_lo":       ll_lo[c_idx],
            "clip_hi":       ll_hi[c_idx],
            "clip_rate":     per_class_clip_rate[c_idx],
            "precision":     cls_pt["precision"][c_idx],
            "precision_lo":  prec_lo[c_idx],
            "precision_hi":  prec_hi[c_idx],
            "recall":        cls_pt["recall"][c_idx],
            "recall_lo":     rec_lo[c_idx],
            "recall_hi":     rec_hi[c_idx],
            "f1":            cls_pt["f1"][c_idx],
            "f1_lo":         f1_lo[c_idx],
            "f1_hi":         f1_hi[c_idx],
        })

    # ---- MACRO_AVG summary row -----------------------------------------
    rows.append({
        "class":         "MACRO_AVG",
        "n_c":           int(per_class_n.sum()),
        "accuracy":      float(np.nanmean(per_class_acc)),
        "acc_lo":        np.nan,
        "acc_hi":        np.nan,
        "brier":         float(np.nanmean(per_class_brier)),
        "brier_lo":      np.nan,
        "brier_hi":      np.nan,
        "clipped_ll":    float(np.nanmean(per_class_ll)),
        "clip_lo":       np.nan,
        "clip_hi":       np.nan,
        "clip_rate":     float(np.nanmean(per_class_clip_rate)),
        "precision":     cls_pt["macro_precision"],
        "precision_lo":  macro_p_lo,
        "precision_hi":  macro_p_hi,
        "recall":        cls_pt["macro_recall"],
        "recall_lo":     macro_r_lo,
        "recall_hi":     macro_r_hi,
        "f1":            cls_pt["macro_f1"],
        "f1_lo":         macro_f_lo,
        "f1_hi":         macro_f_hi,
    })

    # ---- WEIGHTED_AVG summary row --------------------------------------
    rows.append({
        "class":         "WEIGHTED_AVG",
        "n_c":           int(per_class_n.sum()),
        "accuracy":      np.nan,
        "acc_lo":        np.nan,
        "acc_hi":        np.nan,
        "brier":         np.nan,
        "brier_lo":      np.nan,
        "brier_hi":      np.nan,
        "clipped_ll":    np.nan,
        "clip_lo":       np.nan,
        "clip_hi":       np.nan,
        "clip_rate":     np.nan,
        "precision":     cls_pt["weighted_precision"],
        "precision_lo":  weighted_p_lo,
        "precision_hi":  weighted_p_hi,
        "recall":        cls_pt["weighted_recall"],
        "recall_lo":     weighted_r_lo,
        "recall_hi":     weighted_r_hi,
        "f1":            cls_pt["weighted_f1"],
        "f1_lo":         weighted_f_lo,
        "f1_hi":         weighted_f_hi,
    })

    # ---- POPULATION row (Hajek-weighted) -------------------------------
    pop_acc   = float((correct * w).sum() / w.sum())
    pop_brier = float((b * w).sum() / w.sum())
    pop_ll    = float((ll * w).sum() / w.sum())
    pop_clip  = float((clipped.astype(float) * w).sum() / w.sum())

    boot_pop_acc   = bootstrap_population_metric(correct.astype(float), w, boot_idx)
    boot_pop_brier = bootstrap_population_metric(b, w, boot_idx)
    boot_pop_ll    = bootstrap_population_metric(ll, w, boot_idx)
    p_acc_lo,   p_acc_hi   = percentile_ci(boot_pop_acc[:, None]);   p_acc_lo, p_acc_hi = float(p_acc_lo[0]), float(p_acc_hi[0])
    p_b_lo,     p_b_hi     = percentile_ci(boot_pop_brier[:, None]); p_b_lo,   p_b_hi   = float(p_b_lo[0]),   float(p_b_hi[0])
    p_ll_lo,    p_ll_hi    = percentile_ci(boot_pop_ll[:, None]);    p_ll_lo,  p_ll_hi  = float(p_ll_lo[0]),  float(p_ll_hi[0])

    # Population-level kappa is the unweighted kappa on the full eval set;
    # we store its bootstrap CI in the "precision_lo/hi" slot of the
    # POPULATION row is not quite right semantically, so we don't reuse the
    # P/R/F1 slots here. Instead we expose kappa separately in the
    # cross-target summary.
    rows.append({
        "class":         "POPULATION",
        "n_c":           int(round(n_eff(w))),
        "accuracy":      pop_acc,
        "acc_lo":        p_acc_lo,
        "acc_hi":        p_acc_hi,
        "brier":         pop_brier,
        "brier_lo":      p_b_lo,
        "brier_hi":      p_b_hi,
        "clipped_ll":    pop_ll,
        "clip_lo":       p_ll_lo,
        "clip_hi":       p_ll_hi,
        "clip_rate":     pop_clip,
        "precision":     np.nan,
        "precision_lo":  np.nan,
        "precision_hi":  np.nan,
        "recall":        np.nan,
        "recall_lo":     np.nan,
        "recall_hi":     np.nan,
        "f1":            np.nan,
        "f1_lo":         np.nan,
        "f1_hi":         np.nan,
    })

    # ---- UNIFORM_BASELINE row ------------------------------------------
    rows.append({
        "class":         "UNIFORM_BASELINE",
        "n_c":           np.nan,
        "accuracy":      1.0 / n_Y,
        "acc_lo":        np.nan,
        "acc_hi":        np.nan,
        "brier":         1.0 - 1.0 / n_Y,
        "brier_lo":      np.nan,
        "brier_hi":      np.nan,
        "clipped_ll":    -np.log(n_Y),
        "clip_lo":       np.nan,
        "clip_hi":       np.nan,
        "clip_rate":     np.nan,
        # For a uniform-random classifier the per-class precision/recall/F1
        # is 1/n_Y. Macro/weighted F1 is also 1/n_Y in expectation.
        "precision":     1.0 / n_Y,
        "precision_lo":  np.nan,
        "precision_hi":  np.nan,
        "recall":        1.0 / n_Y,
        "recall_lo":     np.nan,
        "recall_hi":     np.nan,
        "f1":            1.0 / n_Y,
        "f1_lo":         np.nan,
        "f1_hi":         np.nan,
    })

    df = pd.DataFrame(rows).set_index("class")

    # Tuck the population-level kappa point estimate + CI onto the DataFrame
    # as attributes so the cross-target summary can pull them without
    # re-running the bootstrap.
    df.attrs["kappa"]        = float(kappa_pt)
    df.attrs["kappa_lo"]     = kappa_lo
    df.attrs["kappa_hi"]     = kappa_hi
    df.attrs["macro_f1"]     = cls_pt["macro_f1"]
    df.attrs["macro_f1_lo"]  = macro_f_lo
    df.attrs["macro_f1_hi"]  = macro_f_hi
    df.attrs["weighted_f1"]  = cls_pt["weighted_f1"]
    df.attrs["weighted_f1_lo"] = weighted_f_lo
    df.attrs["weighted_f1_hi"] = weighted_f_hi

    return df


# Compute for every target, both kinds.
fidelity_tables = {}     # (target, kind) -> DataFrame
target_summary_attrs = {}  # (target, kind) -> dict of summary scalars (kappa, macro_f1, ...)
for t, run in runs.items():
    for kind in run["prob_kinds"]:
        print(f"Computing {t} ({kind})... ", end="", flush=True)
        tbl = compute_fidelity_table(
            run["scored"], t, run["classes"], kind,
            B=B_BOOT, seed=GLOBAL_SEED,
        )
        fidelity_tables[(t, kind)] = tbl
        target_summary_attrs[(t, kind)] = dict(tbl.attrs)
        out_path = METRICS_DIR / f"fidelity_{t}_{kind}.csv"
        tbl.to_csv(out_path)
        print(f"saved -> {out_path.name}")


In [ ]:
# Display headline (debiased) tables for each target.
print("=" * 80)
print("PER-TARGET FIDELITY TABLES (debiased probabilities)")
print("=" * 80)
print("Rows below the per-class block:")
print("  MACRO_AVG    : unweighted mean of precision / recall / F1 across classes")
print("  WEIGHTED_AVG : true-class-frequency-weighted mean of P / R / F1")
print("  POPULATION   : Hajek-weighted population mean of accuracy / brier / clipped_ll")
print("  UNIFORM_BASELINE : 1/n_Y reference")
print()

for t in runs:
    kind = "deb" if (t, "deb") in fidelity_tables else "raw"
    tbl = fidelity_tables[(t, kind)].copy()
    attrs = target_summary_attrs[(t, kind)]
    print(f"\n[{t}]  kind={kind}  "
          f"kappa={attrs['kappa']:.3f}  "
          f"[{attrs['kappa_lo']:.3f}, {attrs['kappa_hi']:.3f}]")

    # Probability metrics block
    print("  Probability metrics:")
    show_prob = tbl[["n_c", "accuracy", "acc_lo", "acc_hi",
                     "brier", "brier_lo", "brier_hi",
                     "clipped_ll", "clip_rate"]].copy()
    print(show_prob.to_string(float_format=lambda x: f"{x:.3f}"))

    # Argmax classification metrics block
    print("  Classification metrics:")
    show_cls = tbl[["precision", "precision_lo", "precision_hi",
                    "recall",    "recall_lo",    "recall_hi",
                    "f1",        "f1_lo",        "f1_hi"]].copy()
    print(show_cls.to_string(float_format=lambda x: f"{x:.3f}"))


## 5. Cross-target summary

One row per target showing the population-level metric values plus
$N_{\text{eff}}$ and uniform baseline. This is the table you'd put in
the report's main results section.


In [ ]:
summary_rows = []
for t in runs:
    kind = "deb" if (t, "deb") in fidelity_tables else "raw"
    tbl  = fidelity_tables[(t, kind)]
    attrs = target_summary_attrs[(t, kind)]
    pop  = tbl.loc["POPULATION"]
    base = tbl.loc["UNIFORM_BASELINE"]
    summary_rows.append({
        "target":            t,
        "n_classes":         len(runs[t]["classes"]),
        "kind":              kind,
        "N_eff":             int(pop["n_c"]),
        "pop_accuracy":      pop["accuracy"],
        "pop_acc_ci":        f"[{pop['acc_lo']:.3f}, {pop['acc_hi']:.3f}]",
        "baseline_acc":      base["accuracy"],
        "pop_brier":         pop["brier"],
        "pop_brier_ci":      f"[{pop['brier_lo']:.3f}, {pop['brier_hi']:.3f}]",
        "baseline_brier":    base["brier"],
        "skill_brier":       1.0 - pop["brier"] / base["brier"],
        "pop_clipped_ll":    pop["clipped_ll"],
        "baseline_ll":       base["clipped_ll"],
        # New: chance-corrected agreement + macro / weighted F1
        "kappa":             attrs["kappa"],
        "kappa_ci":          f"[{attrs['kappa_lo']:.3f}, {attrs['kappa_hi']:.3f}]",
        "kappa_lo":          attrs["kappa_lo"],
        "kappa_hi":          attrs["kappa_hi"],
        "macro_f1":          attrs["macro_f1"],
        "macro_f1_ci":       f"[{attrs['macro_f1_lo']:.3f}, {attrs['macro_f1_hi']:.3f}]",
        "macro_f1_lo":       attrs["macro_f1_lo"],
        "macro_f1_hi":       attrs["macro_f1_hi"],
        "weighted_f1":       attrs["weighted_f1"],
        "weighted_f1_ci":    f"[{attrs['weighted_f1_lo']:.3f}, {attrs['weighted_f1_hi']:.3f}]",
        "weighted_f1_lo":    attrs["weighted_f1_lo"],
        "weighted_f1_hi":    attrs["weighted_f1_hi"],
    })

cross_target = pd.DataFrame(summary_rows).set_index("target")

print("CROSS-TARGET POPULATION-LEVEL SUMMARY")
print("=" * 80)
print()
print("Probability metrics:")
print(cross_target[["n_classes", "kind", "N_eff",
                    "pop_accuracy", "pop_acc_ci",
                    "pop_brier",    "pop_brier_ci",
                    "skill_brier",
                    "pop_clipped_ll"]].to_string(float_format=lambda x: f"{x:.3f}"))
print()
print("Classification metrics (chance-corrected and F1 family):")
print(cross_target[["kappa", "kappa_ci",
                    "macro_f1", "macro_f1_ci",
                    "weighted_f1", "weighted_f1_ci"]].to_string(float_format=lambda x: f"{x:.3f}"))

cross_target_path = METRICS_DIR / "cross_target_summary.csv"
cross_target.to_csv(cross_target_path)
print(f"\nSaved -> {cross_target_path}")


## 6. Figure: per-class Brier with CIs

One panel per target. Each panel shows per-class Brier as a horizontal
bar with bootstrap CI as an error range; uniform baseline is shown as a
vertical reference line. Classes are ordered by point estimate so the
visual story (which classes the model handles well vs poorly) reads
top-to-bottom.


In [ ]:
n_targets = len(runs)
ncols = 2
nrows = (n_targets + ncols - 1) // ncols

fig, axes = plt.subplots(
    nrows=nrows, ncols=ncols,
    figsize=(7 * ncols, 2.5 + 1.0 * max(len(r["classes"]) for r in runs.values())),
    squeeze=False,
)

for i, t in enumerate(runs):
    ax = axes[i // ncols, i % ncols]
    kind = "deb" if (t, "deb") in fidelity_tables else "raw"
    tbl = fidelity_tables[(t, kind)]
    classes_only = tbl.drop(["POPULATION", "UNIFORM_BASELINE"]).copy()
    classes_only = classes_only.sort_values("brier", ascending=True)

    y_pos = np.arange(len(classes_only))
    pe    = classes_only["brier"].values
    lo    = classes_only["brier_lo"].values
    hi    = classes_only["brier_hi"].values
    err   = np.array([pe - lo, hi - pe])

    ax.errorbar(
        pe, y_pos,
        xerr=err, fmt="o", color="#3a6f9b", ecolor="#7fa6c4",
        capsize=3, markersize=5, linewidth=1.2,
    )

    # Uniform baseline
    base_brier = 1.0 - 1.0 / len(runs[t]["classes"])
    ax.axvline(base_brier, color="#c44e52", linestyle="--", linewidth=1.0,
               label=f"uniform baseline ({base_brier:.2f})")

    # Population point estimate
    pop_brier = tbl.loc["POPULATION", "brier"]
    ax.axvline(pop_brier, color="#55a868", linestyle=":", linewidth=1.2,
               label=f"population ({pop_brier:.3f})")

    ax.set_yticks(y_pos)
    ax.set_yticklabels(classes_only.index, fontsize=9)
    ax.set_xlabel("Brier score (lower = better)")
    ax.set_title(f"{t}  ({kind}, n_Y={len(runs[t]['classes'])})", fontsize=11)
    ax.set_xlim(0, max(2.0, base_brier * 1.1))
    ax.legend(loc="upper right", fontsize=8, frameon=True)
    ax.grid(axis="x", linestyle=":", alpha=0.4)
    ax.spines[["top", "right"]].set_visible(False)

# Hide unused subplots
for j in range(n_targets, nrows * ncols):
    axes[j // ncols, j % ncols].set_visible(False)

fig.suptitle(
    "Per-class Brier scores with 95% bootstrap CIs (debiased)",
    fontsize=13, y=1.00,
)
plt.tight_layout()
plt.savefig(METRICS_DIR / "fig1_brier_per_class.png",
            dpi=180, bbox_inches="tight")
plt.savefig(METRICS_DIR / "fig1_brier_per_class.pdf",
            bbox_inches="tight")
plt.show()


## 7. Raw-vs-debiased ablation

Two-column table: population-level Brier and accuracy under raw vs
debiased probabilities, plus the absolute and percentage shift. This is
the appendix table that demonstrates PriDe's effect on headline metrics
(not just on the letter-bias diagnostic).

**Interpretation.** Letter bias on the cross-tabs translates into a
change in headline metrics only if the bias correlates with content. A
target where PriDe reduces letter bias by 80% but barely moves Brier
(e.g. RACE) tells you the bias was orthogonal to content; a target where
PriDe materially shifts Brier (e.g. SEX) tells you the bias was
content-relevant.


In [ ]:
ablation_rows = []
for t in runs:
    if (t, "raw") not in fidelity_tables or (t, "deb") not in fidelity_tables:
        continue
    raw_pop = fidelity_tables[(t, "raw")].loc["POPULATION"]
    deb_pop = fidelity_tables[(t, "deb")].loc["POPULATION"]
    ablation_rows.append({
        "target":         t,
        "n_classes":      len(runs[t]["classes"]),
        "brier_raw":      raw_pop["brier"],
        "brier_deb":      deb_pop["brier"],
        "brier_delta":    deb_pop["brier"] - raw_pop["brier"],
        "brier_pct":      100.0 * (deb_pop["brier"] - raw_pop["brier"])
                          / raw_pop["brier"] if raw_pop["brier"] > 0 else np.nan,
        "acc_raw":        raw_pop["accuracy"],
        "acc_deb":        deb_pop["accuracy"],
        "acc_delta":      deb_pop["accuracy"] - raw_pop["accuracy"],
    })

ablation = pd.DataFrame(ablation_rows).set_index("target")
print("RAW vs DEBIASED ABLATION (population-level)")
print("=" * 80)
print(ablation.to_string(float_format=lambda x: f"{x:.4f}"))

ablation_path = METRICS_DIR / "raw_vs_deb_ablation.csv"
ablation.to_csv(ablation_path)
print(f"\nSaved -> {ablation_path}")
